## Evaluating AbLMs on the 10 donor test sets

Cross-entropy loss of one model on each of the 10 donor datasets. The mean over the ten is the
value that goes into `isodata.csv` and into the S3 Table; `02-plots.ipynb` fits the FixedData
profiles to those means.

The donor datasets are `Fixed_data_profiles.zip` on
[Zenodo](https://zenodo.org/records/16938681); unzip it into `data/`. Run this notebook once per
model, with `model_path` set to that model (all fifteen are on the
[Hugging Face hub](https://huggingface.co/collections/brineylab/ablms-scaling-laws)).

In [ ]:
import glob

import numpy as np

from datasets import load_dataset
from transformers import (
    EsmTokenizer,
    EsmForMaskedLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

In [ ]:
# the ten donor test sets
donor_files = sorted(glob.glob('./data/Fixed_data_profiles/*.csv'))
len(donor_files)

In [ ]:
# Load the tokenizer and model
# the ESM-2 vocabulary is the same for every size, so one tokenizer works for all of them
tokenizer = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")

model_path = 'brineylab/150M_full_checkpoint-500000'   # or a local checkpoint directory
model = EsmForMaskedLM.from_pretrained(model_path)

In [ ]:
# Define parameters for tokenization
MAX_LEN = 320  # e.g. train_config["max_length"]
SEPARATOR = "<cls><cls>"  # e.g. train_config["separator_token"]

In [ ]:
# Tokenize the dataset
def preprocess_function(example):
    # Combine the heavy and light chain sequences
    sequence = example['sequence_aa_heavy'] + SEPARATOR + example['sequence_aa_light']
    # Tokenize with the same settings used during training
    tokenized = tokenizer(
        sequence,
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        add_special_tokens=True,
    )
    # Add special tokens mask if required (as done in training)
    tokenized['special_tokens_mask'] = tokenizer.get_special_tokens_mask(tokenized['input_ids'], already_has_special_tokens=True)
    return tokenized

In [ ]:
# Prepare the data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15  # Use the same probability as in training
)

# Setup evaluation arguments
training_args = TrainingArguments(
    output_dir='./results',
    per_device_eval_batch_size=32,
    logging_dir='./logs',
    do_eval=True,
    evaluation_strategy="no",
    seed=42,
    report_to="none",  # Explicitly disable W&B logging
)

In [ ]:
# Run the evaluation on every donor
losses = []
for path in donor_files:
    dataset = load_dataset('csv', data_files={'test': path})
    eval_dataset = dataset.map(preprocess_function, batched=False)['test']

    trainer = Trainer(
        model=model,
        args=training_args,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )
    loss = trainer.evaluate()['eval_loss']
    losses.append(loss)
    print(f"{path}: {loss:.5f}")

# the average cross-entropy per token over the ten donors, as reported in the S3 Table
print(f"\nmean over {len(losses)} donors: {np.mean(losses):.5f}")